# Part 4: Demo App Walkthrough
## Agentic Security Hands-On Lab

**Estimated Time:** 20 minutes

**Prerequisites:** Complete Parts 1, 2, & 3

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Log in to the AskHR demo app** using SSO via IBM Verify
2. **Query the HR agent** as a basic-access user and observe what data is returned
3. **Observe access control enforcement** — a salary query is denied for a user without admin privileges
4. **Switch to an admin user** and observe that the same salary query now succeeds
5. **Understand how group membership in IBM Verify drives Vault policy and database access**

---

## 📋 Prerequisites

Before starting this notebook, ensure:

- ✅ All Docker services are running (`langflow`, `postgres`, `vault`, `frontend`, `backend`)
- ✅ The AskHR Agent flow has been imported and configured in Langflow (Part 3)
- ✅ The demo frontend is accessible at http://localhost:3000

---

## 🌐 Step 1: Open the Demo Frontend

Open your browser and navigate to:

```
http://localhost:3000
```

You should see the **AskHR** application login page.

---

## 🔐 Step 2: Log In as Alice Smith (Basic Access User)

### Login Instructions:

1. Click the **"Login with SSO"** button
2. You will be redirected to the **IBM Verify** login page
3. Make sure to select **"Cloud Directory"** as the login method
4. Enter the following credentials:
   - **Username:** `alice.smith`
   - **Password:** *(provided by your instructor)*
5. Click **"Sign In"**
6. You will be redirected back to the AskHR app, now logged in as Alice

> 💡 **Who is Alice?** Alice belongs to the `hr-basic` group in IBM Verify. This group is mapped to the `hr-basic` Vault policy, which only grants access to basic employee information — **not** salary data.

---

## 🔍 Step 3: Query Employee Information (Alice)

### Test 1: Get All Employees

In the chat input, type:

```
Show me all employees
```

**Expected result:** The agent should return a list of employees with basic information (name, department, job title, etc.).

This works because Alice's `hr-basic` role has read access to the general employee table.

---

### Test 2: Get Employee Info for Sarah Johnson

In the chat input, type:

```
Get employee information for Sarah Johnson
```

> 📝 Sarah Johnson is employee ID `1` in the database.

**Expected result:** The agent should return Sarah's basic employee profile — name, department, job title, and other non-sensitive fields.

---

## 🚫 Step 4: Attempt a Salary Query (Alice — Access Denied)

Now let's try to access salary information, which Alice is **not** authorised to view.

In the chat input, type:

```
What is Sarah Johnson's salary?
```

**Expected result:** The agent should respond with something like:

> *"The request for salary information for Sarah Johnson has failed due to a permission denied error. The system was unable to retrieve the necessary credentials to access the database, resulting in the failure of the query."*

### Why did this happen?

Here is the full security enforcement chain that led to this denial:

1. **Alice logs in** → IBM Verify issues an access token containing her group membership (`hr-basic`)
2. **Token Exchange** → The agent exchanges Alice's token for a Vault-scoped token using OAuth 2.0 Token Exchange
3. **Vault Policy Check** → Vault evaluates Alice's token against the `hr-basic` policy, which does **not** permit access to the salary secrets path
4. **Credential Request Denied** → Vault refuses to issue database credentials for the salary-access role
5. **Query Fails** → Without valid credentials, the database tool cannot execute the salary query
6. **Agent Reports Failure** → The LLM surfaces the permission error in a natural language response

> 🔒 Notice that the access control is enforced **at the infrastructure level** (Vault), not just in application logic. Even if the agent tried to bypass the check, it would still fail — because it never receives the credentials needed to run the query.

---

## 🔄 Step 5: Switch Users — Log In as Bob Anderson (Admin User)

Now we'll repeat the salary query as an admin user to see the difference.

### Logout Instructions:

1. Click your **profile / avatar** in the top right corner of the app
2. Click **"Logout"**
3. **Close the browser tab (or the entire browser window)** to fully clear the session
4. Open a **new browser window** and navigate to http://localhost:3000

> ⚠️ It is important to close and reopen the browser to ensure the previous SSO session is fully cleared. Simply clicking logout may not be sufficient if the browser retains the IBM Verify session cookie.

### Login as Bob:

1. Click **"Login with SSO"**
2. Select **"Cloud Directory"** as the login method
3. Enter the following credentials:
   - **Username:** `bob.anderson`
   - **Password:** *(provided by your instructor)*
4. Click **"Sign In"**

> 💡 **Who is Bob?** Bob belongs to the `hr-admin` group in IBM Verify. This group is mapped to the `hr-admin` Vault policy, which grants access to **both** basic employee data and salary information.

---

## ✅ Step 6: Retry the Salary Query (Bob — Access Granted)

Now that you are logged in as Bob, run the same salary query:

```
What is Sarah Johnson's salary?
```

**Expected result:** The agent should now successfully return Sarah Johnson's salary information. The response will look something like:

> *"The salary information for employee ID 1 is as follows:*
>
> *Employee ID: 1*  
> *First Name: Sarah*  
> *Last Name: Johnson*  
> *Department: Engineering*  
> *Annual Salary: $145,000.00*  
> *Bonus Percentage: 15.00%*  
> *Stock Options: 5,000*  
> *Salary Grade: E5*  
> *Performance Rating: Exceeds Expectations*  
>
> *Please note that this information is confidential and should only be accessed by authorized personnel."*

### Why does this work now?

The same security chain runs — but this time:

1. **Bob logs in** → IBM Verify issues a token with Bob's group membership (`hr-admin`)
2. **Token Exchange** → The agent exchanges Bob's token for a Vault-scoped token
3. **Vault Policy Check** → Vault evaluates Bob's token against the `hr-admin` policy, which **permits** access to the salary secrets path
4. **Credentials Issued** → Vault issues dynamic database credentials scoped to the `hr-admin` role
5. **Query Succeeds** → The database tool connects using those credentials and retrieves the salary data
6. **Agent Responds** → The LLM formats the result into a natural language response

> 🔑 The only thing that changed was **who logged in**. The agent code, the flow, and the database are identical. Access control is entirely driven by the user's identity and group membership — enforced dynamically through Vault at runtime.

---

## 🔍 Step 7: Review Agent Logs in Langflow (Optional)

Now that you have run both scenarios, go back to Langflow at http://localhost:7860 and inspect the agent logs to see exactly what happened under the hood.

For each run, you can review:

- **Agent reasoning**: Which tools the agent decided to call and in what order
- **Token Exchange output**: The scoped token returned by IBM Verify
- **Vault tool output**: Whether credentials were issued or denied, and for which role
- **Database tool output**: The raw query result (or the error message)
- **LLM output**: The final response generated from the tool results

Compare the logs from Alice's failed salary query vs. Bob's successful one — you will see exactly where in the chain the access decision was made.

---

## ✅ Summary

In this notebook, you:

1. ✅ Logged in to the AskHR demo app using SSO via IBM Verify (Cloud Directory)
2. ✅ Queried employee information as Alice (basic access) — succeeded
3. ✅ Attempted a salary query as Alice — **denied** due to `hr-basic` Vault policy
4. ✅ Logged in as Bob (admin access) and retried the salary query — **succeeded** due to `hr-admin` Vault policy
5. ✅ Observed that access control is enforced at the infrastructure level, not in application code

---

## 🎯 Key Takeaways

- **Identity-driven access**: The user's IBM Verify group membership determines what data the agent can access — no hardcoded permissions in the app
- **Infrastructure-level enforcement**: Vault enforces policy before credentials are ever issued — the agent cannot bypass this
- **Dynamic credentials**: Database credentials are generated on-demand per request and scoped to the user's role — they are never stored or reused
- **Transparent to the user**: The agent handles the entire security flow invisibly — the user simply asks a question and receives an appropriate response

---